# Phishing Detector Starter Notebook

Catalog files, build labels, parse emails, de‑duplicate, and run **Leave‑One‑Dataset‑Out (LODO)** baselines.
> Expectation: You place your CSV files under `data/raw/`. Update `label_map.csv` as needed.

## Environment & Paths

In [1]:

import os, re, json, math, glob, hashlib, email, html, string, itertools, random
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import pandas as pd
import numpy as np

try:
    from bs4 import BeautifulSoup
    _HAS_BS4 = True
except Exception:
    _HAS_BS4 = False

# Project paths (relative to this notebook)
DATA_RAW = Path("./data/raw")
OUT_DIR = Path(".")
ART_DIR = OUT_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)

LABEL_MAP_PATH = OUT_DIR / "label_map.csv"
if LABEL_MAP_PATH.exists():
    label_map_df = pd.read_csv(LABEL_MAP_PATH)
    display(label_map_df.head())
else:
    display("label_map.csv not found — create it or copy from your project root.")


,dataset_key,file_name,assumed_label_raw,mapped_binary_label,notes
0,CEAS_08,CEAS_08.csv,spam,not_phish,Non-phishing spam/benchmark
1,Enron,Enron.csv,ham,not_phish,Legitimate ham corpus
2,Ling,Ling.csv,spam,not_phish,Non-phishing spam/benchmark
3,Nazario,Nazario.csv,phish,phish,Explicit phishing corpus
4,Nazario_5,Nazario_5.csv,phish,phish,Explicit phishing corpus


In [2]:
# Auto-build label_map.csv from the files present in ./data/raw
import re
from pathlib import Path
import pandas as pd

# If your notebook lives in phishing_detection/, this is correct:
DATA_RAW = Path("./data/raw")
assert DATA_RAW.exists(), f"Folder not found: {DATA_RAW.resolve()}"

def dataset_key_from_fname(p: Path) -> str:
    # CEAS_08.csv  -> CEAS_08
    # TREC_07.csv  -> TREC_07
    # Enron.csv    -> Enron
    return re.sub(r"\.csv(\.gz)?$", "", p.name, flags=re.I)

# Heuristic mapper: phish vs not_phish
def heuristic_map(dk: str):
    dk_low = dk.lower()
    if any(tok in dk_low for tok in ["nazario", "nigerian", "419", "phish"]):
        return {"label_raw": "phish", "mapped": "phish", "notes": "Explicit phishing corpus"}
    if any(tok in dk_low for tok in ["enron", "ham"]):
        return {"label_raw": "ham", "mapped": "not_phish", "notes": "Legitimate ham corpus"}
    if any(tok in dk_low for tok in ["spamassasin", "spamassassin", "trec", "ceas", "ling", "spam"]):
        return {"label_raw": "spam", "mapped": "not_phish", "notes": "Non-phishing spam/benchmark"}
    return {"label_raw": "unknown", "mapped": "unknown", "notes": "Please verify"}

files = sorted(list(DATA_RAW.glob("*.csv")) + list(DATA_RAW.glob("*.csv.gz")))
rows = []
for p in files:
    dk = dataset_key_from_fname(p)
    info = heuristic_map(dk)
    rows.append({
        "dataset_key": dk,
        "file_name": p.name,
        "assumed_label_raw": info["label_raw"],
        "mapped_binary_label": info["mapped"],
        "notes": info["notes"]
    })

label_map_df = pd.DataFrame(rows)
label_map_df.to_csv("label_map.csv", index=False)  # writes alongside the notebook
label_map_df


,dataset_key,file_name,assumed_label_raw,mapped_binary_label,notes
0,CEAS_08,CEAS_08.csv,spam,not_phish,Non-phishing spam/benchmark
1,Enron,Enron.csv,ham,not_phish,Legitimate ham corpus
2,Ling,Ling.csv,spam,not_phish,Non-phishing spam/benchmark
3,Nazario,Nazario.csv,phish,phish,Explicit phishing corpus
4,Nazario_5,Nazario_5.csv,phish,phish,Explicit phishing corpus
5,Nigerian_5,Nigerian_5.csv,phish,phish,Explicit phishing corpus
6,Nigerian_Fraud,Nigerian_Fraud.csv,phish,phish,Explicit phishing corpus
7,SpamAssasin,SpamAssasin.csv,spam,not_phish,Non-phishing spam/benchmark
8,TREC_05,TREC_05.csv,spam,not_phish,Non-phishing spam/benchmark
9,TREC_06,TREC_06.csv,spam,not_phish,Non-phishing spam/benchmark


## Utilities: URL extraction, HTML → text, RFC822 parsing, normalization

In [3]:

URL_RE = re.compile(r"\b((?:https?://|www\.)[^\s<>\"']+)\b", re.IGNORECASE)

def extract_urls(text: str) -> List[str]:
    if not isinstance(text, str) or not text:
        return []
    return [u.rstrip(').,;:') for u in URL_RE.findall(text)]

def html_to_text(html_str: str) -> str:
    if not isinstance(html_str, str) or not html_str:
        return ""
    if _HAS_BS4:
        try:
            soup = BeautifulSoup(html_str, "html.parser")
            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()
            return soup.get_text(" ", strip=True)
        except Exception:
            pass
    s = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", html_str, flags=re.I|re.S)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def parse_email(raw: str) -> Tuple[str, str, str]:
    if not isinstance(raw, str):
        raw = str(raw)
    try:
        msg = email.message_from_string(raw)
        subj = msg.get('Subject') or ""
        body_text, body_html = "", ""
        if msg.is_multipart():
            for part in msg.walk():
                ctype = (part.get_content_type() or "").lower()
                disp = (part.get('Content-Disposition') or "").lower()
                if "attachment" in disp:
                    continue
                try:
                    payload = part.get_payload(decode=True)
                    if payload is None:
                        continue
                    charset = part.get_content_charset() or "utf-8"
                    text = payload.decode(charset, errors="replace")
                except Exception:
                    continue
                if ctype == "text/plain":
                    body_text += "\n" + text
                elif ctype == "text/html":
                    body_html += "\n" + text
        else:
            payload = msg.get_payload(decode=True)
            if payload is None:
                text = msg.get_payload()
            else:
                charset = msg.get_content_charset() or "utf-8"
                text = payload.decode(charset, errors="replace")
            if isinstance(text, str) and ("<html" in text.lower() or "</p>" in text.lower()):
                body_html = text or ""
            else:
                body_text = text or ""
        if body_html and not body_text:
            body_text = html_to_text(body_html)
        return (subj or "").strip(), (body_text or "").strip(), html_to_text(body_html)
    except Exception:
        return "", raw, ""

def normalize_text(t: str) -> str:
    t = (t or "").lower()
    t = html.unescape(t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def text_signature(subject: str, body: str) -> str:
    norm = normalize_text(subject + " " + body)
    return hashlib.md5(norm.encode("utf-8")).hexdigest()

def jaccard_similarity(a: set, b: set) -> float:
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union if union else 0.0

def char_shingles(text: str, k: int = 5) -> set:
    text = normalize_text(text)
    if len(text) < k: return {text}
    return {text[i:i+k] for i in range(len(text)-k+1)}


## Load & Parse CSVs

Walk `data/raw/`, infer the likely text column, parse email, extract URLs, and map labels from `label_map.csv`.

In [4]:
# === Load & Parse CSVs (robust, self-contained) ===
# Drop this in one cell. It defines safe_read_csv + loader + helpers and produces all_df.

import re, csv, email, html
from pathlib import Path
from typing import List, Optional
import pandas as pd
import numpy as np

# 0) Paths (adjust if your notebook sits elsewhere)
DATA_RAW = Path("./data/raw")

# 1) URL extraction + lightweight HTML->text + RFC822 parsing
URL_RE = re.compile(r"\b((?:https?://|www\.)[^\s<>\"']+)\b", re.IGNORECASE)

def extract_urls(text: str) -> List[str]:
    if not isinstance(text, str) or not text:
        return []
    return [u.rstrip(').,;:') for u in URL_RE.findall(text)]

def html_to_text(html_str: str) -> str:
    if not isinstance(html_str, str) or not html_str:
        return ""
    s = re.sub(r"<(script|style)[^>]*>.*?</\1>", " ", html_str, flags=re.I|re.S)
    s = re.sub(r"<[^>]+>", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def parse_email(raw: str):
    """Return (subject, body_text, body_html_text). Fallbacks if not RFC822."""
    if not isinstance(raw, str):
        raw = str(raw)
    try:
        msg = email.message_from_string(raw)
        subj = msg.get('Subject') or ""
        body_text, body_html = "", ""
        if msg.is_multipart():
            for part in msg.walk():
                ctype = (part.get_content_type() or "").lower()
                disp = (part.get('Content-Disposition') or "").lower()
                if "attachment" in disp:
                    continue
                try:
                    payload = part.get_payload(decode=True)
                    if payload is None:
                        continue
                    charset = part.get_content_charset() or "utf-8"
                    text = payload.decode(charset, errors="replace")
                except Exception:
                    continue
                if ctype == "text/plain":
                    body_text += "\n" + text
                elif ctype == "text/html":
                    body_html += "\n" + text
        else:
            payload = msg.get_payload(decode=True)
            if payload is None:
                text = msg.get_payload()
            else:
                charset = msg.get_content_charset() or "utf-8"
                text = payload.decode(charset, errors="replace")
            if isinstance(text, str) and ("<html" in text.lower() or "</p>" in text.lower()):
                body_html = text or ""
            else:
                body_text = text or ""
        if body_html and not body_text:
            body_text = html_to_text(body_html)
        return (subj or "").strip(), (body_text or "").strip(), html_to_text(body_html)
    except Exception:
        return "", raw, ""

# 2) Robust CSV reader for messy files
def safe_read_csv(path: Path) -> pd.DataFrame:
    trials = [
        dict(engine="c",     sep=",",   encoding="utf-8",   on_bad_lines="skip"),
        dict(engine="python",sep=",",   encoding="utf-8",   on_bad_lines="skip"),
        dict(engine="python",sep=None,  encoding="utf-8",   on_bad_lines="skip"),  # auto-detect
        dict(engine="python",sep=",",   encoding="latin-1", on_bad_lines="skip"),
        dict(engine="python",sep=None,  encoding="latin-1", on_bad_lines="skip"),
    ]
    last_err = None
    for opts in trials:
        try:
            return pd.read_csv(
                path,
                dtype=str,
                keep_default_na=False,
                na_values=[],
                low_memory=False,
                quoting=csv.QUOTE_MINIMAL,
                escapechar="\\",
                **opts
            )
        except Exception as e:
            last_err = e
            continue
    raise last_err

# 3) Helpers for picking columns / dataset key
TEXT_CANDIDATE_COLS = [
    "text","Text","message","Message","raw","Raw","email","Email","content","Content",
    "body","Body","mail","Mail","data","Data"
]
SUBJECT_CANDIDATE_COLS = ["subject","Subject","SUBJECT","subj","Subj"]

def pick_first_existing(candidates: List[str], cols: List[str]) -> Optional[str]:
    for c in candidates:
        if c in cols:
            return c
    return None

def dataset_key_from_fname(path: Path) -> str:
    return re.sub(r"\.csv(\.gz)?$", "", path.name, flags=re.I)

# 4) The loader: uses label_map_df from your earlier cell
def load_all_raw(data_dir: Path, label_map_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    files = sorted(data_dir.glob("*.csv")) + sorted(data_dir.glob("*.csv.gz"))
    if not files:
        print("No CSVs found in", data_dir)
    for f in files:
        dk = dataset_key_from_fname(f)
        lm = label_map_df[label_map_df["dataset_key"] == dk] if not label_map_df.empty else pd.DataFrame()
        if lm.empty:
            print(f"[skip] {f.name}: dataset_key not in label_map.csv")
            continue
        mapped = lm.iloc[0]["mapped_binary_label"]
        if mapped not in ("phish","not_phish"):
            print(f"[skip] {f.name}: mapped label is {mapped}")
            continue
        try:
            df = safe_read_csv(f)
        except Exception as e:
            print(f"[skip] {f.name}: could not parse ({e})")
            continue

        text_col = pick_first_existing(TEXT_CANDIDATE_COLS, list(df.columns))
        subj_col = pick_first_existing(SUBJECT_CANDIDATE_COLS, list(df.columns))
        if text_col is None:
            # fallback: pick column with the largest mean length
            text_col = max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())

        for _, r in df.iterrows():
            raw = str(r.get(text_col, ""))
            subj_raw = str(r.get(subj_col, "")) if subj_col else ""
            subj, body_txt, body_html_txt = parse_email(raw if len(raw) > len(subj_raw) else subj_raw + "\n" + raw)
            subj = subj or subj_raw or ""
            urls = extract_urls(subj + " " + body_txt)
            rows.append({
                "dataset_key": dk,
                "file_name": f.name,
                "subject": subj,
                "body_text": body_txt if body_txt else raw,
                "label": mapped,
                "urls": urls
            })
    return pd.DataFrame(rows)

# 5) Run it
assert 'label_map_df' in globals() and not label_map_df.empty, "Run the label-map cell first to create label_map_df."
all_df = load_all_raw(DATA_RAW, label_map_df)
print("Loaded rows:", len(all_df))
display(all_df.head())


[skip] TREC_05.csv: could not parse (The 'low_memory' option is not supported with the 'python' engine)
[skip] TREC_06.csv: could not parse (The 'low_memory' option is not supported with the 'python' engine)
Loaded rows: 145635


,dataset_key,file_name,subject,body_text,label,urls
0,CEAS_08,CEAS_08.csv,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",not_phish,[http://whitedone.com]
1,CEAS_08,CEAS_08.csv,Befriend Jenna Jameson,Upgrade your sex and pleasures with these tech...,not_phish,[http://www.brightmade.com]
2,CEAS_08,CEAS_08.csv,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,not_phish,[http://www.cnn.com/video/partners/email/index...
3,CEAS_08,CEAS_08.csv,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,not_phish,[http://en.wikipedia.org/wiki/.so_%28domain_na...
4,CEAS_08,CEAS_08.csv,SpecialPricesPharmMoreinfo,WelcomeFastShippingCustomerSupport\nhttp://7iw...,not_phish,[http://7iwfna.blu.livefilestore.com/y1pXdX3kw...


## De-duplication

Exact duplicates removal; optional near‑dup via Jaccard over char 5‑grams (off by default).

In [5]:

# === Drop-in replacement: robust dedup (no external deps) ===
import re, hashlib, numpy as np, pandas as pd

def _normalize_text(t: str) -> str:
    t = (t or "").lower()
    t = re.sub(r"\s+", " ", t)
    return t.strip()

def _md5_sig(text: str) -> str:
    return hashlib.md5(_normalize_text(text).encode("utf-8")).hexdigest()

def dedup(df: pd.DataFrame, near_dup: bool=False, jaccard_threshold: float=0.95) -> pd.DataFrame:
    if df.empty:
        return df
    # Ensure required columns exist
    if "subject" not in df.columns: df = df.assign(subject="")
    if "body_text" not in df.columns: df = df.assign(body_text="")
    # Exact-duplicate signature on (subject + body_text)
    combined = df["subject"].astype(str).str.cat(df["body_text"].astype(str), sep=" ")
    sigs = combined.map(_md5_sig)
    df = df.assign(_sig=sigs)
    df = df.drop_duplicates(subset=["_sig"]).drop(columns=["_sig"])
    if not near_dup:
        return df

    # Optional near-duplicate filter (requires char_shingles + jaccard_similarity defined elsewhere)
    kept = []
    for dk, g in df.groupby("dataset_key"):
        g = g.copy()
        shingles_list = [char_shingles((r.subject or "") + " " + (r.body_text or "")) for r in g.itertuples(index=False)]
        keep_mask = np.ones(len(g), dtype=bool)
        for i in range(len(g)):
            if not keep_mask[i]:
                continue
            for j in range(i+1, len(g)):
                if not keep_mask[j]:
                    continue
                sim = jaccard_similarity(shingles_list[i], shingles_list[j])
                if sim >= jaccard_threshold:
                    keep_mask[j] = False
        kept.append(g[keep_mask])
    return pd.concat(kept, ignore_index=True)


dedup_df = dedup(all_df, near_dup=False)
print("After exact-dup removal:", len(dedup_df))
display(dedup_df.head())


After exact-dup removal: 136686


,dataset_key,file_name,subject,body_text,label,urls
0,CEAS_08,CEAS_08.csv,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",not_phish,[http://whitedone.com]
1,CEAS_08,CEAS_08.csv,Befriend Jenna Jameson,Upgrade your sex and pleasures with these tech...,not_phish,[http://www.brightmade.com]
2,CEAS_08,CEAS_08.csv,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,not_phish,[http://www.cnn.com/video/partners/email/index...
3,CEAS_08,CEAS_08.csv,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,not_phish,[http://en.wikipedia.org/wiki/.so_%28domain_na...
4,CEAS_08,CEAS_08.csv,SpecialPricesPharmMoreinfo,WelcomeFastShippingCustomerSupport\nhttp://7iw...,not_phish,[http://7iwfna.blu.livefilestore.com/y1pXdX3kw...


## Leave‑One‑Dataset‑Out (LODO) splits

Hold out one dataset for testing; stratified train/val on the rest.

In [6]:

from sklearn.model_selection import train_test_split

def lodo_splits(df: pd.DataFrame, test_dataset: str, val_size: float=0.1, seed: int=42):
    df = df.copy()
    test_df = df[df["dataset_key"] == test_dataset]
    rest_df = df[df["dataset_key"] != test_dataset]
    if rest_df.empty or test_df.empty:
        raise ValueError("Not enough data for LODO with test_dataset=" + test_dataset)
    X = rest_df.index.values
    y = (rest_df["label"] == "phish").astype(int).values
    train_idx, val_idx = train_test_split(X, test_size=val_size, random_state=seed, stratify=y)
    train_df = rest_df.loc[train_idx]
    val_df = rest_df.loc[val_idx]
    return train_df, val_df, test_df


## Baseline model: char TF‑IDF + Logistic Regression

Train per LODO fold; save models and a results CSV.

In [9]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

def join_text(df: pd.DataFrame):
    return (df["subject"].fillna("") + " \n " + df["body_text"].fillna("")).tolist()

def train_eval_baseline(train_df, val_df, test_df, fold_name="fold"):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=200000)),
        ("lr", LogisticRegression(max_iter=400, class_weight="balanced"))
    ])
    y_tr = (train_df["label"]=="phish").astype(int).values
    y_va = (val_df["label"]=="phish").astype(int).values
    y_te = (test_df["label"]=="phish").astype(int).values

    pipe.fit(join_text(train_df), y_tr)
    val_probs = pipe.predict_proba(join_text(val_df))[:,1]
    test_probs = pipe.predict_proba(join_text(test_df))[:,1]

    metrics = {
        "val_roc_auc": float(roc_auc_score(y_va, val_probs)) if len(np.unique(y_va))>1 else float("nan"),
        "val_pr_auc": float(average_precision_score(y_va, val_probs)) if len(np.unique(y_va))>1 else float("nan"),
        "test_roc_auc": float(roc_auc_score(y_te, test_probs)) if len(np.unique(y_te))>1 else float("nan"),
        "test_pr_auc": float(average_precision_score(y_te, test_probs)) if len(np.unique(y_te))>1 else float("nan"),
    }
    print(f"[{fold_name}] Validation ROC-AUC: {metrics['val_roc_auc']:.4f}, PR-AUC: {metrics['val_pr_auc']:.4f}")
    print(f"[{fold_name}] Test ROC-AUC: {metrics['test_roc_auc']:.4f}, PR-AUC: {metrics['test_pr_auc']:.4f}")
    return pipe, metrics

def run_lodo(df: pd.DataFrame, deduped: bool=True):
    work = dedup(df) if deduped else df
    datasets = sorted(work["dataset_key"].unique())
    results = []
    for holdout in datasets:
        try:
            tr, va, te = lodo_splits(work, holdout)
        except Exception as e:
            print(f"[skip fold {holdout}] {e}")
            continue
        model, metrics = train_eval_baseline(tr, va, te, fold_name=holdout)
        import joblib, os
        model_path = ART_DIR / f"baseline_tfidf_lr__holdout_{holdout}.joblib"
        joblib.dump(model, model_path)
        metrics.update({
            "holdout_dataset": holdout,
            "n_train": len(tr),
            "n_val": len(va),
            "n_test": len(te),
            "model_path": str(model_path)
        })
        results.append(metrics)
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_csv = OUT_DIR / "lodo_results.csv"
        res_df.to_csv(res_csv, index=False)
        print("Saved LODO results to", res_csv)
    return res_df

# Example (uncomment after data is loaded into data/raw/):
#lodo_df = run_lodo(dedup_df, deduped=True)
#display(lodo_df)


In [11]:
# ==== Paired LODO: hold out one phish dataset + one not_phish dataset together ====
import pandas as pd
from sklearn.model_selection import train_test_split

def run_lodo_paired(df: pd.DataFrame, label_map_df: pd.DataFrame, deduped: bool=True, seed: int=42):
    work = dedup(df) if deduped else df

    # Figure out which dataset_keys are phish vs not_phish from the label map
    lm = label_map_df[["dataset_key", "mapped_binary_label"]].drop_duplicates()
    phish_keys = sorted(lm[lm["mapped_binary_label"]=="phish"]["dataset_key"].unique().tolist())
    nonphish_keys = sorted(lm[lm["mapped_binary_label"]=="not_phish"]["dataset_key"].unique().tolist())

    results = []
    for p in phish_keys:
        for n in nonphish_keys:
            holdout_pair = {p, n}
            test_df = work[work["dataset_key"].isin(holdout_pair)].copy()
            rest_df = work[~work["dataset_key"].isin(holdout_pair)].copy()

            # ensure both classes exist in test (they should, but double-check)
            if test_df["label"].nunique() < 2:
                print(f"[skip pair {p}|{n}] test set is single-class")
                continue

            # stratified train/val on the remainder
            X = rest_df.index.values
            y = (rest_df["label"]=="phish").astype(int).values
            try:
                tr_idx, va_idx = train_test_split(X, test_size=0.1, random_state=seed, stratify=y)
            except ValueError as e:
                print(f"[skip pair {p}|{n}] stratify failed on remainder: {e}")
                continue

            train_df = rest_df.loc[tr_idx]
            val_df   = rest_df.loc[va_idx]

            model, metrics = train_eval_baseline(train_df, val_df, test_df, fold_name=f"{p}|{n}")
            metrics.update({
                "holdout_pair": f"{p}|{n}",
                "n_train": len(train_df),
                "n_val": len(val_df),
                "n_test": len(test_df),
            })
            results.append(metrics)

    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df.to_csv("lodo_paired_results.csv", index=False)
        print("Saved paired LODO results to lodo_paired_results.csv")
    return res_df

# Example run:
aired_df = run_lodo_paired(dedup_df, label_map_df, deduped=True)
display(paired_df)


[Nazario|CEAS_08] Validation ROC-AUC: 0.9874, PR-AUC: 0.8911
[Nazario|CEAS_08] Test ROC-AUC: 0.7086, PR-AUC: 0.0711
[Nazario|Enron] Validation ROC-AUC: 0.9872, PR-AUC: 0.8897
[Nazario|Enron] Test ROC-AUC: 0.6735, PR-AUC: 0.0679
[Nazario|Ling] Validation ROC-AUC: 0.9888, PR-AUC: 0.8833
[Nazario|Ling] Test ROC-AUC: 0.9868, PR-AUC: 0.9725
[Nazario|SpamAssasin] Validation ROC-AUC: 0.9917, PR-AUC: 0.9163
[Nazario|SpamAssasin] Test ROC-AUC: 0.3543, PR-AUC: 0.1678
[skip pair Nazario|TREC_05] test set is single-class
[skip pair Nazario|TREC_06] test set is single-class
[Nazario|TREC_07] Validation ROC-AUC: 0.9956, PR-AUC: 0.9538
[Nazario|TREC_07] Test ROC-AUC: 0.4856, PR-AUC: 0.0249
[Nazario_5|CEAS_08] Validation ROC-AUC: 0.9916, PR-AUC: 0.9261
[Nazario_5|CEAS_08] Test ROC-AUC: 0.7980, PR-AUC: 0.2319
[Nazario_5|Enron] Validation ROC-AUC: 0.9885, PR-AUC: 0.9099
[Nazario_5|Enron] Test ROC-AUC: 0.7711, PR-AUC: 0.1234


KeyboardInterrupt: 

## Per‑dataset error analysis (optional)

After training a specific holdout model, inspect top false negatives/positives to guide feature additions.

In [ ]:

def error_table(model, test_df: pd.DataFrame, k: int=20):
    y_true = (test_df["label"]=="phish").astype(int).values
    probs = model.predict_proba(join_text(test_df))[:,1]
    preds = (probs>=0.5).astype(int)
    test_df = test_df.copy()
    test_df["prob_phish"] = probs
    test_df["pred"] = preds
    fns = test_df[(test_df["label"]=="phish") & (test_df["pred"]==0)].sort_values("prob_phish", ascending=True).head(k)
    fps = test_df[(test_df["label"]=="not_phish") & (test_df["pred"]==1)].sort_values("prob_phish", ascending=False).head(k)
    display({"false_negatives": fns[["dataset_key","subject","prob_phish"]],
             "false_positives": fps[["dataset_key","subject","prob_phish"]]})
